# RNNs and LSTMs

## Objectives
- Understand recurrent neural networks
- LSTM and GRU architectures
- Vanishing/exploding gradients
- Sequence-to-sequence models
- Bidirectional RNNs

## Introduction
RNNs process sequences by maintaining hidden state across timesteps. LSTMs address vanishing gradients with gate mechanisms, enabling long-range dependencies.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 1. Simple RNN

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden):
        combined = torch.cat([x, hidden], 1)
        hidden = torch.tanh(self.i2h(combined))
        output = self.h2o(hidden)
        return output, hidden
    
    def init_hidden(self, batch_size=1):
        return torch.zeros(batch_size, self.hidden_size)

model = SimpleRNN(input_size=10, hidden_size=20, output_size=5)
hidden = model.init_hidden(batch_size=2)
x = torch.randn(2, 10)
output, hidden = model(x, hidden)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden state shape: {hidden.shape}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


In [ ]:
# Define a custom function with detailed implementation
## 2. LSTM Architecture

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
    
    def forward(self, x):
        out, (hidden, cell) = self.lstm(x)
        return out, hidden, cell

print("""
LSTM GATES:
1. Input Gate: Controls how much new info enters cell state
2. Forget Gate: Controls how much old info to discard
3. Cell Gate: New candidate values to add
4. Output Gate: Controls output based on cell state

Equations:
- Forget: f_t = σ(W_f·[h_{t-1}, x_t] + b_f)
- Input: i_t = σ(W_i·[h_{t-1}, x_t] + b_i)
- Cell: C̃_t = tanh(W_c·[h_{t-1}, x_t] + b_c)
- Output: o_t = σ(W_o·[h_{t-1}, x_t] + b_o)
- Cell state: C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
- Hidden: h_t = o_t ⊙ tanh(C_t)
""")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Define a custom function with detailed implementation
## 3. GRU (Gated Recurrent Unit)

class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
    
    def forward(self, x):
        out, hidden = self.gru(x)
        return out, hidden

print("""
GRU vs LSTM:

GRU (2 gates):
- Simpler, fewer parameters
- Faster training
# Iterate through batches of data
- Good for smaller datasets

LSTM (3 gates):
- More expressive
# Iterate through batches of data
- Better for complex sequences
- Handles long dependencies better
- More parameters (3×LSTM vs 2×GRU)

When to use:
- GRU: Start here, fast prototyping
- LSTM: Complex tasks, more data
""")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 4. Bidirectional RNN

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, bidirectional=True, batch_first=True)
# Iterate through batches of data
        self.output = nn.Linear(hidden_size * 2, hidden_size)  # 2× for bidirectional
    
    def forward(self, x):
        out, (hidden, cell) = self.lstm(x)
        # out shape: (batch, seq_len, hidden_size*2)
        return out, hidden, cell

model = BiLSTM(input_size=10, hidden_size=20)
x = torch.randn(2, 5, 10)  # (batch, seq_len, input_size)
out, hidden, cell = model(x)
print(f"Bidirectional LSTM:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape} (seq_len × (2×hidden_size))")
print(f"Processes sequence forward AND backward")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 5. Vanishing/Exploding Gradients

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vanishing gradients
steps = np.arange(0, 50)
vanishing = 1.0 * (0.9 ** steps)  # decay over time
exploding = 1.0 * (1.1 ** steps)  # grow over time
lstm_stable = np.ones_like(steps) * 0.99

axes[0].semilogy(steps, vanishing, label='RNN (Vanishing)', linewidth=2.5)
axes[0].semilogy(steps, lstmstable, label='LSTM (Stable)', linewidth=2.5)
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Gradient Magnitude (log scale)')
axes[0].set_title('Vanishing Gradient Problem')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Exploding gradients
axes[1].semilogy(steps, exploding, label='RNN (Exploding)', linewidth=2.5)
axes[1].semilogy(steps, lstmstable, label='LSTM (Stable)', linewidth=2.5)
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Gradient Magnitude (log scale)')
axes[1].set_title('Exploding Gradient Problem')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([1e-3, 1e3])

plt.tight_layout()
plt.show()

print("\nSolutions to gradient problems:")
print("- LSTM gates maintain stable gradient flow")
print("- Gradient clipping prevents exploding gradients")
print("- Layer normalization stabilizes training")
print("- Skip connections (ResNets) help gradients flow")


In [ ]:
# Define a custom function with detailed implementation
## 6. Sequence-to-Sequence (Seq2Seq)

class Seq2Seq(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(output_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, src, trg):
        # Encoder
        _, (hidden, cell) = self.encoder(src)
        
        # Decoder (uses encoder hidden state)
        outputs = []
# Iterate through batches of data
        for t in range(trg.size(1)):
            output, (hidden, cell) = self.decoder(trg[:, t:t+1], (hidden, cell))
            pred = self.fc(output)
            outputs.append(pred)
        
        return torch.cat(outputs, dim=1)

print("""
SEQ2SEQ ARCHITECTURE:

ENCODER:
- Processes input sequence
- Produces context vector (hidden state)
- Compresses information

DECODER:
- Uses context vector as initial state
- Generates output sequence
- One timestep at a time

Applications:
- Machine translation (English → French)
- Summarization
- Question answering
- Image captioning

Example: "I love cats" → "J'aime les chats"
""")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 7. Training RNN

def train_rnn_epoch(model, data, optimizer, criterion):
    model.train()
# Compute the loss (error) between predictions and actual values
    total_loss = 0
    
# Iterate through batches of data
    for inputs, targets in data:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
# Compute the loss (error) between predictions and actual values
        loss = criterion(outputs.view(-1, 5), targets.view(-1))
        
        # Backward pass
        loss.backward()
        # Gradient clipping
# Update model parameters based on computed gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
# Compute the loss (error) between predictions and actual values
        total_loss += loss.item()
    
    return total_loss / len(data)

print("""
RNN TRAINING TIPS:

1. GRADIENT CLIPPING:
# Update model parameters based on computed gradients
   torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
   Prevents exploding gradients

2. LEARNING RATE:
   Use lower LR (0.001-0.01) than CNNs
   More sensitive to large updates

3. INITIALIZATION:
# Iterate through batches of data
   Use orthogonal initialization for RNN weights
   Better gradient flow

4. SEQUENCE LENGTH:
   Truncate very long sequences
   Computational and gradient considerations

5. BATCH NORMALIZATION:
   Careful with RNNs (can disrupt temporal dependencies)
   Use layer normalization instead
""")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 8. Comparison and Use Cases

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Architecture comparison
models = ['RNN', 'GRU', 'LSTM', 'BiLSTM']
# Update model parameters based on computed gradients
params = [100, 150, 250, 500]
accuracy = [75, 82, 88, 91]

axes[0].bar(models, accuracy, color=['blue', 'green', 'orange', 'red'], alpha=0.7)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('RNN Architecture Comparison')
axes[0].set_ylim([70, 95])
axes[0].grid(True, alpha=0.3, axis='y')
# Iterate through batches of data
for i, (m, a) in enumerate(zip(models, accuracy)):
    axes[0].text(i, a + 0.5, f"{a}%", ha='center')

# Sequence length handling
seq_lengths = [10, 50, 100, 200, 500]
rnn_acc = [85, 72, 45, 20, 5]  # Deteriorates with length
lstm_acc = [88, 86, 85, 84, 82]  # Maintains performance

axes[1].plot(seq_lengths, rnn_acc, marker='o', label='RNN', linewidth=2.5, markersize=8)
axes[1].plot(seq_lengths, lstm_acc, marker='s', label='LSTM', linewidth=2.5, markersize=8)
axes[1].set_xlabel('Sequence Length')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Performance vs Sequence Length')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Set up the neural network model architecture
## Key Takeaways
- RNNs process sequences with recurrent connections maintaining state
- LSTMs use gates to control information flow and prevent gradient problems
- GRUs are simpler alternative to LSTMs with fewer parameters
- Bidirectional RNNs consider context from both directions
- Vanishing/exploding gradients require LSTM cells and gradient clipping
# Iterate through batches of data
- Seq2Seq (encoder-decoder) for sequence-to-sequence tasks
- Gradient clipping is essential for stable RNN training

## Interview Q&A

# Iterate through batches of data
**Q1: Why is LSTM better than RNN for long sequences?**
RNNs suffer from vanishing gradients - gradients decay exponentially backward through time, preventing learning from long-range dependencies. LSTMs use forget gates and cell states to maintain gradient flow, enabling learning from sequences of 100+ timesteps.

**Q2: What does the LSTM forget gate do?**
The forget gate decides what information from the previous cell state to discard. It outputs values between 0-1: 0 means forget completely, 1 means keep completely. This allows the model to reset when encountering new unrelated information.

**Q3: When would you use bidirectional RNN?**
# Iterate through batches of data
When you need context from both past and future (not real-time constraints). Examples: text classification, NER, machine translation (non-autoregressive). Not suitable for prediction tasks where future is unknown.

## References
- [LSTM Paper](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [GRU Paper](https://arxiv.org/abs/1406.1078)
